# Sentio Bias Classifier — QLoRA Fine-tuning

**Model**: `Qwen/Qwen2.5-3B-Instruct` (teacher-student distillation from Claude Haiku)

**Task**: Multi-label cognitive bias detection in first-person journal entries (15 classes)

**Technique**: QLoRA (4-bit NF4 quantization + LoRA adapters, rank 16)

**Pipeline**:
1. Generate 750 labeled training examples via Claude Haiku (teacher, ~$0.15)
2. Fine-tune Qwen2.5-3B student with SFTTrainer + W&B tracking
3. Head-to-head eval: student vs Claude Haiku on 30-entry human-labeled holdout
4. Push adapter to HuggingFace Hub + production integration guide

**Hardware**: Kaggle T4 GPU (16 GB VRAM). Training time: ~35–45 min.

**Required Kaggle Secrets** (Settings → Add-ons → Secrets):
- `ANTHROPIC_API_KEY` — for data generation and eval baseline
- `HF_TOKEN` — for pushing adapter to HF Hub
- `WANDB_API_KEY` — for experiment tracking (optional; set to `off` to skip)

**Cost estimate**: ~\$0.20 total Claude API credits.

In [ ]:
# Install dependencies — pinned for reproducibility on Kaggle
# Run this cell first; kernel restart is NOT needed (Kaggle handles it)
!pip install -q \
    "transformers>=4.45.0,<5.0" \
    "peft>=0.13.0,<0.15" \
    "trl>=0.8.6,<0.10" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.34.0" \
    "datasets>=2.20.0" \
    "anthropic>=0.40.0" \
    "wandb>=0.18.0" \
    "scikit-learn>=1.3.0" \
    "huggingface_hub>=0.24.0"

print("Installation complete.")

In [ ]:
import subprocess, sys, torch

# Verify GPU is available
assert torch.cuda.is_available(), "GPU not available — enable GPU accelerator in Kaggle settings"

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {gpu_mem_gb:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

# Qwen2.5-3B in 4-bit needs ~4 GB VRAM for weights + ~6 GB for training
if gpu_mem_gb < 14:
    print("WARNING: Less than 14 GB VRAM detected. Reduce per_device_train_batch_size to 1.")

In [ ]:
import os, json, re, time, logging, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ── Kaggle Secrets ──────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    ANTHROPIC_KEY = _sec.get_secret("ANTHROPIC_API_KEY")
    HF_TOKEN      = _sec.get_secret("HF_TOKEN")
    WANDB_KEY     = _sec.get_secret("WANDB_API_KEY")
    print("Secrets loaded from Kaggle vault")
except Exception:
    ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY", "")
    HF_TOKEN      = os.getenv("HF_TOKEN", "")
    WANDB_KEY     = os.getenv("WANDB_API_KEY", "")
    print("Secrets loaded from environment variables")

assert ANTHROPIC_KEY, "ANTHROPIC_API_KEY secret is missing — add it in Kaggle Settings → Secrets"

os.environ["ANTHROPIC_API_KEY"]  = ANTHROPIC_KEY
os.environ["HF_TOKEN"]           = HF_TOKEN
os.environ["WANDB_API_KEY"]      = WANDB_KEY
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # suppress HF tokenizer warnings

# ── Paths & constants ───────────────────────────────────────────────────────
WORK_DIR    = Path("/kaggle/working")
DATA_PATH   = WORK_DIR / "bias_training.jsonl"
OUTPUT_DIR  = WORK_DIR / "qlora-adapter"
RESULTS_PATH = WORK_DIR / "eval_results.json"

BASE_MODEL  = "Qwen/Qwen2.5-3B-Instruct"
HF_REPO_ID  = "sentio-bias-qlora-qwen25-3b"   # pushed as <your-hf-username>/<HF_REPO_ID>
N_PER_BIAS  = 50    # examples per class; 15 × 50 = 750 total
SEED        = 42

BIAS_LABELS = [
    "confirmation_bias", "attribution_error", "all_or_nothing",
    "catastrophizing",   "mind_reading",      "overgeneralization",
    "emotional_reasoning", "should_statements", "labeling",
    "personalization",   "availability_bias", "anchoring_bias",
    "dunning_kruger",    "sunk_cost_fallacy", "fundamental_attribution",
]
LABEL2ID = {b: i for i, b in enumerate(BIAS_LABELS)}
N_CLASSES = len(BIAS_LABELS)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{N_CLASSES} bias classes | base model: {BASE_MODEL}")

---
## Phase 1 — Data Generation (Teacher: Claude Haiku)

Claude Haiku is the production bias classifier (the teacher).
We ask it to **generate + label** 50 realistic journal entries per bias class.

This is teacher-student distillation: the student (Qwen2.5-3B) learns to mimic
the teacher's knowledge, at a fraction of the inference cost.

Design choices:
- **Prompt caching**: the 400-token system prompt is cached via `cache_control: ephemeral`
  → tokens billed only once per 5-min window
- **Journal realism**: 80–200 word entries in first-person; bias embedded in reasoning,
  not stated explicitly
- **Co-occurring biases**: each entry may have 0-1 secondary biases; multi-label ground truth
- **Retry logic**: 3 attempts per bias class with exponential backoff

In [ ]:
import anthropic

BIAS_DESCRIPTIONS = {
    "confirmation_bias":      "Seeking or interpreting information that confirms pre-existing beliefs while ignoring contradictory evidence.",
    "attribution_error":      "Attributing others' behavior to character flaws while attributing one's own to circumstances (Fundamental Attribution Error variant).",
    "all_or_nothing":         "Seeing situations in black-and-white terms with no middle ground.",
    "catastrophizing":        "Assuming the worst possible outcome will occur from a situation.",
    "mind_reading":           "Assuming you know what others are thinking without evidence.",
    "overgeneralization":     "Drawing broad conclusions from a single event (uses 'always', 'never', 'everyone', 'no one').",
    "emotional_reasoning":    "Treating feelings as facts ('I feel stupid, therefore I am stupid').",
    "should_statements":      "Rigid rules about how oneself or others must behave ('I should', 'they must', 'I have to').",
    "labeling":               "Reducing oneself or others to a single negative trait ('I'm a failure', 'he's an idiot').",
    "personalization":        "Taking excessive personal responsibility for external events.",
    "availability_bias":      "Overweighting recent or easily recalled events when making judgments.",
    "anchoring_bias":         "Over-relying on the first piece of information encountered when making decisions.",
    "dunning_kruger":         "Overestimating one's competence in areas where one has limited knowledge.",
    "sunk_cost_fallacy":      "Continuing a course of action because of past investment rather than future value.",
    "fundamental_attribution": "Underweighting situational factors when judging others' behavior.",
}

GEN_SYSTEM_PROMPT = """You are a clinical psychologist and cognitive behavioral therapist with expertise in cognitive biases and distortions. You generate realistic, psychologically accurate journal entry examples for training a cognitive bias detection AI model.

Guidelines for generated entries:
- Write in first-person journal style (casual, reflective, personal — like a real diary)
- Length: 80–200 words per entry
- The bias should be embedded naturally in the reasoning, NOT stated or named explicitly
- Vary the life contexts: work performance, interpersonal conflicts, self-evaluation, decision-making, social situations
- Language should feel authentic and human — not overly literary, clinical, or perfect
- Each entry has ONE primary bias; it may also contain 0–1 secondary biases
- Vary the demographic implied by the text: student, professional, parent, etc.

Return ONLY a valid JSON array. No markdown code fences, no explanation text."""


def _generate_for_bias(client: anthropic.Anthropic, bias: str, n: int) -> list[dict]:
    """Call Claude Haiku to generate n journal entry examples for one bias class."""
    desc = BIAS_DESCRIPTIONS[bias]
    user_prompt = (
        f"Generate {n} diverse journal entry examples that naturally exhibit **{bias}**:\n"
        f"Definition: {desc}\n\n"
        f"Return a JSON array with exactly {n} elements. Each element:\n"
        f"[\n"
        f"  {{\n"
        f"    \"text\": \"journal entry text here (80-200 words)\",\n"
        f"    \"primary_bias\": \"{bias}\",\n"
        f"    \"co_occurring\": [],\n"
        f"    \"context\": \"work|relationships|decisions|self|social\"\n"
        f"  }}\n"
        f"]\n\n"
        f"Requirements:\n"
        f"- Exactly {n} elements in the array\n"
        f"- \"text\" must be 80–200 words\n"
        f"- \"co_occurring\" is a list of secondary bias_ids from the same 15-class taxonomy (usually empty or 1 item)\n"
        f"- \"primary_bias\" must always be exactly \"{bias}\"\n"
        f"- Vary \"context\" across examples"
    )

    for attempt in range(3):
        try:
            resp = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=8192,
                system=[{
                    "type": "text",
                    "text": GEN_SYSTEM_PROMPT,
                    "cache_control": {"type": "ephemeral"},  # cache the 400-token system prompt
                }],
                messages=[{"role": "user", "content": user_prompt}],
            )
            raw = resp.content[0].text.strip()
            # Strip markdown fences Haiku sometimes adds despite instructions
            raw = re.sub(r"^```(?:json)?\n?", "", raw)
            raw = re.sub(r"\n?```$", "", raw)
            raw = raw.strip()
            examples = json.loads(raw)
            if not isinstance(examples, list) or len(examples) == 0:
                raise ValueError(f"Expected non-empty list, got {type(examples)}")
            # Enforce correct primary_bias label in case Haiku made a typo
            for ex in examples:
                ex["primary_bias"] = bias
                # Validate co_occurring references valid bias IDs only
                ex["co_occurring"] = [
                    b for b in ex.get("co_occurring", []) if b in LABEL2ID and b != bias
                ]
            cache_hits = getattr(resp.usage, "cache_read_input_tokens", 0)
            log.info(f"  {bias}: {len(examples)} examples (cache_read={cache_hits})")
            return examples
        except Exception as exc:
            log.warning(f"  {bias} attempt {attempt+1}/3 failed: {exc}")
            if attempt < 2:
                time.sleep(5 * (attempt + 1))
    log.error(f"  {bias}: all attempts failed — skipping")
    return []


def generate_dataset(output_path: Path, n_per_bias: int = 50) -> int:
    """Generate training data for all 15 bias classes. Returns total examples written."""
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    total = 0
    with open(output_path, "w", encoding="utf-8") as f:
        for i, bias in enumerate(BIAS_LABELS):
            log.info(f"[{i+1}/{N_CLASSES}] Generating {n_per_bias} examples for '{bias}'...")
            examples = _generate_for_bias(client, bias, n_per_bias)
            for ex in examples:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
            total += len(examples)
            time.sleep(2)  # stay within Haiku rate limits (50k tokens/min)
    return total


# Only generate if data doesn't already exist (idempotent)
if DATA_PATH.exists():
    with open(DATA_PATH) as f:
        existing_n = sum(1 for _ in f)
    print(f"Training data already exists: {existing_n} examples at {DATA_PATH}")
    print("Delete the file and re-run this cell to regenerate.")
else:
    print(f"Generating {N_PER_BIAS * N_CLASSES} examples ({N_PER_BIAS} per class × {N_CLASSES} classes)...")
    print("Estimated time: 15–20 min | Cost: ~$0.12 in Claude Haiku credits")
    n_total = generate_dataset(DATA_PATH, n_per_bias=N_PER_BIAS)
    print(f"\nDataset generated: {n_total} examples → {DATA_PATH}")

In [ ]:
import pandas as pd
from collections import Counter

# ── Load & validate ─────────────────────────────────────────────────────────
records = []
malformed = 0
with open(DATA_PATH, encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        try:
            row = json.loads(line.strip())
        except json.JSONDecodeError:
            malformed += 1
            continue
        # Required fields
        if not all(k in row for k in ("text", "primary_bias")):
            malformed += 1
            continue
        if row["primary_bias"] not in LABEL2ID:
            malformed += 1
            continue
        text = str(row["text"]).strip()
        word_count = len(text.split())
        if word_count < 20:  # too short to be useful
            malformed += 1
            continue
        records.append({
            "text": text,
            "primary_bias": row["primary_bias"],
            "co_occurring": [b for b in row.get("co_occurring", []) if b in LABEL2ID],
            "context": row.get("context", "unknown"),
            "word_count": word_count,
        })

df = pd.DataFrame(records)

print(f"Total records   : {len(df)}")
print(f"Malformed/skipped: {malformed}")
print(f"Word count       : min={df.word_count.min()} | mean={df.word_count.mean():.0f} | max={df.word_count.max()}")
print()

# Class distribution
dist = df.primary_bias.value_counts()
print("Class distribution:")
for bias in BIAS_LABELS:
    count = dist.get(bias, 0)
    bar = "█" * (count // 2)
    print(f"  {bias:<30s} {count:>4d}  {bar}")

# Multilabel stats
multi = df[df.co_occurring.map(len) > 0]
print(f"\nMulti-label entries: {len(multi)} ({100*len(multi)/len(df):.1f}%)")

# Sanity: check each class has enough examples
min_examples = dist.min()
assert min_examples >= 10, f"Class '{dist.idxmin()}' has only {min_examples} examples — regenerate"
print(f"\nValidation passed: all {N_CLASSES} classes have ≥{min_examples} examples.")

In [ ]:
# Show one sample entry per class for qualitative inspection
print("=" * 70)
print("SAMPLE ENTRIES (one per class)")
print("=" * 70)
for bias in BIAS_LABELS:
    sample = df[df.primary_bias == bias].iloc[0]
    print(f"\n[{bias}]")
    print(f"  Context : {sample.context}")
    print(f"  Words   : {sample.word_count}")
    # Print first 120 chars of text
    preview = sample.text[:120].replace("\n", " ")
    print(f"  Text    : {preview}...")

---
## Phase 2 — QLoRA Fine-tuning

**Architecture choices**:

| Decision | Choice | Rationale |
|:---------|:-------|:----------|
| Base model | Qwen2.5-3B-Instruct | Fully open (no access gate), strong reasoning, built-in chat template |
| Quantization | 4-bit NF4 (BitsAndBytes) | Fits 16 GB T4; NF4 is information-theoretically optimal for normally-distributed weights |
| LoRA rank | 16 | Balance between parameter count (~40M trainable) and training stability |
| Target modules | all linear projections | Full-rank equivalent in the low-rank subspace |
| Training objective | Causal LM (instruction tuning) | Teaches the model to generate structured JSON output |
| Loss masking | Response-only via SFT | Prompt tokens get label=-100; loss computed only on assistant output |

**Instruction format** (Qwen ChatML template):
```
<|im_start|>system
You are a cognitive bias detection system...
<|im_end|>
<|im_start|>user
<entry>journal text here</entry>
<|im_end|>
<|im_start|>assistant
{"biases": ["confirmation_bias"], "confidence": 0.88}
<|im_end|>
```

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ── 4-bit NF4 quantization config ───────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4: optimal quantization for normal-distributed weights
    bnb_4bit_use_double_quant=True,     # double quantization saves ~0.4 bits/param extra
    bnb_4bit_compute_dtype=torch.float16,  # compute in fp16 (T4 native; use bfloat16 for A100)
)

print(f"Loading tokenizer: {BASE_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right",   # right-padding for causal LM training
)
# Qwen2.5 uses eos_token as pad_token by default — set explicitly for clarity
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Vocab size: {tokenizer.vocab_size:,} | pad_token: '{tokenizer.pad_token}'")

print(f"\nLoading base model with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",          # automatically assigns layers to GPU/CPU
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False          # must disable for gradient checkpointing
model.config.pretraining_tp = 1         # disable tensor parallelism

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params/1e9:.2f}B total parameters")

# Memory usage after loading
used_gb = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used after loading: {used_gb:.1f} GB")

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

# ── Prepare model for k-bit training (enables gradient checkpointing) ───────
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# ── LoRA configuration ───────────────────────────────────────────────────────
# Target all linear projection layers — equivalent to full fine-tuning in the low-rank subspace
# Qwen2.5 architecture: q/k/v/o projections + MLP gate/up/down projections
lora_config = LoraConfig(
    r=16,                          # rank: 16 adds ~40M trainable params on 3B model (~1.3%)
    lora_alpha=32,                 # scaling factor: alpha/r = 2.0 (standard choice)
    lora_dropout=0.05,             # mild dropout to prevent overfitting on 750 examples
    bias="none",                   # do not add LoRA to bias terms
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # attention projections
        "gate_proj", "up_proj", "down_proj",           # MLP projections
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Sanity check: trainable params should be ~1-2% of total
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B ({100*trainable/total:.2f}%)")

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# ── System prompt for instruction format ─────────────────────────────────────
TAXONOMY_TEXT = "\n".join(
    f"{i+1}. {b} — {BIAS_DESCRIPTIONS[b]}" for i, b in enumerate(BIAS_LABELS)
)

SYSTEM_PROMPT = (
    "You are a cognitive bias detection system. Analyze the journal entry and identify "
    "cognitive biases present in the text.\n\n"
    f"COGNITIVE BIAS TAXONOMY (15 classes):\n{TAXONOMY_TEXT}\n\n"
    "Rules:\n"
    "- Only flag biases clearly evidenced in the text\n"
    "- Most entries have 0–2 biases\n"
    "- Return ONLY valid JSON, no commentary"
)


def _make_label_json(row: dict) -> str:
    """Build the assistant's expected JSON response from training labels."""
    biases = [row["primary_bias"]]
    biases += [b for b in row.get("co_occurring", []) if b in LABEL2ID and b not in biases]
    return json.dumps({"biases": biases, "confidence": 0.85})


def format_for_sft(row: dict) -> dict:
    """Format one training example as a Qwen2.5 chat conversation string."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Analyze this journal entry:\n\n<entry>\n{row['text']}\n</entry>"},
        {"role": "assistant", "content": _make_label_json(row)},
    ]
    # apply_chat_template returns the full formatted string including special tokens
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,  # False = include the assistant turn in output
    )
    return {"text": text, "label": _make_label_json(row)}


# ── Split: 80% train, 10% val, 10% test ─────────────────────────────────────
train_recs, temp_recs = train_test_split(records, test_size=0.2, random_state=SEED,
                                          stratify=[r["primary_bias"] for r in records])
val_recs,  test_recs  = train_test_split(temp_recs, test_size=0.5, random_state=SEED,
                                          stratify=[r["primary_bias"] for r in temp_recs])

print(f"Split: train={len(train_recs)} | val={len(val_recs)} | test={len(test_recs)}")

# Format and create HF Datasets
train_formatted = [format_for_sft(r) for r in train_recs]
val_formatted   = [format_for_sft(r) for r in val_recs]
test_formatted  = [format_for_sft(r) for r in test_recs]

train_hf = Dataset.from_list(train_formatted)
val_hf   = Dataset.from_list(val_formatted)
test_hf  = Dataset.from_list(test_formatted)

print(f"Sample formatted text (first 400 chars):\n")
print(train_hf[0]["text"][:400])
print("...")

# Verify token lengths fit within max_seq_length=512
sample_lengths = [
    len(tokenizer(r["text"], add_special_tokens=False)["input_ids"])
    for r in train_formatted[:100]
]
print(f"\nToken length (first 100 samples): min={min(sample_lengths)} mean={sum(sample_lengths)//len(sample_lengths)} max={max(sample_lengths)}")
if max(sample_lengths) > 512:
    print("WARNING: some sequences exceed 512 tokens — they will be truncated during training.")

In [ ]:
import wandb
from transformers import TrainingArguments
from trl import SFTTrainer

# ── W&B setup ────────────────────────────────────────────────────────────────
use_wandb = bool(WANDB_KEY and WANDB_KEY.lower() != "off")
if use_wandb:
    wandb.login(key=WANDB_KEY, relogin=True)
    wandb.init(
        project="sentio-bias-qlora",
        name="qwen25-3b-qlora-r16",
        config={
            "base_model": BASE_MODEL,
            "n_train": len(train_hf),
            "n_val": len(val_hf),
            "lora_r": 16,
            "lora_alpha": 32,
            "n_classes": N_CLASSES,
            "n_examples_per_class": N_PER_BIAS,
        },
    )
    print("W&B initialized")
else:
    print("W&B disabled (set WANDB_API_KEY secret to enable tracking)")

# ── Training arguments ───────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch size = 2 × 4 = 8
    learning_rate=2e-4,               # standard for LoRA fine-tuning
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,                        # T4 supports fp16; change to bf16=True on A100
    optim="paged_adamw_32bit",        # paged optimizer reduces memory pressure
    max_grad_norm=0.3,                # gradient clipping (important for QLoRA stability)
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,               # keep only the 2 best checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="wandb" if use_wandb else "none",
    run_name="qwen25-3b-qlora-r16",
    dataloader_num_workers=0,         # Kaggle: keep at 0 to avoid multiprocessing issues
    group_by_length=True,             # pad similar-length sequences together → efficiency
)

# ── SFTTrainer ───────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    tokenizer=tokenizer,
    max_seq_length=512,               # truncate to 512 tokens (covers 99%+ of our examples)
    dataset_text_field="text",        # the column SFTTrainer uses as the full chat string
    packing=False,                    # don't pack sequences (small dataset, avoid complexity)
    args=training_args,
)

# ── Train ────────────────────────────────────────────────────────────────────
print("Starting training...")
print(f"  Effective batch size : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Steps per epoch      : {len(train_hf) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

train_result = trainer.train()

print(f"\nTraining complete.")
print(f"  Total steps     : {train_result.global_step}")
print(f"  Training loss   : {train_result.training_loss:.4f}")
print(f"  Runtime         : {train_result.metrics.get('train_runtime', 0)/60:.1f} min")

if use_wandb:
    wandb.log({"train_loss_final": train_result.training_loss})

---
## Phase 3 — Evaluation: Student vs Teacher

Evaluation protocol:
1. **Held-out test set** (10% of generated data, stratified): per-class F1 for the student
2. **30-entry human-labeled holdout**: per-class F1 for both student and Claude Haiku
3. **Head-to-head table**: student vs teacher on the human-labeled set

Metrics: Precision, Recall, F1 per class + macro averages + agreement rate

In [ ]:
# ── Inference helper ─────────────────────────────────────────────────────────
model.eval()

def predict_biases_student(text: str, max_new_tokens: int = 128) -> list[str]:
    """Run the fine-tuned student model on a journal entry. Returns list of bias_ids."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Analyze this journal entry:\n\n<entry>\n{text}\n</entry>"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,               # greedy decoding for deterministic results
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_ids = output_ids[0][inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    # Parse JSON output
    try:
        parsed = json.loads(raw)
        biases = parsed.get("biases", [])
        # Validate: keep only known bias IDs
        return [b for b in biases if b in LABEL2ID]
    except (json.JSONDecodeError, AttributeError):
        # Try to extract any valid bias IDs from the raw string as fallback
        found = [b for b in BIAS_LABELS if b in raw]
        return found[:3]  # cap at 3 to avoid noise


def predict_biases_teacher(text: str) -> list[str]:
    """Run Claude Haiku (production teacher) on a journal entry."""
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    response_format = (
        "Return a JSON array. Each element: "
        '{"bias_id": "<id>", "confidence": <0.5-1.0>}\n'
        "If no biases are detected, return: []"
    )
    try:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{
                "role": "user",
                "content": (
                    f"Analyze this journal entry for cognitive biases:\n"
                    f"<entry>\n{text[:2000]}\n</entry>\n\n{response_format}"
                )
            }]
        )
        raw = resp.content[0].text.strip()
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
        biases_list = json.loads(raw)
        if not isinstance(biases_list, list):
            return []
        return [
            b["bias_id"] for b in biases_list
            if isinstance(b, dict) and b.get("bias_id") in LABEL2ID
            and float(b.get("confidence", 0)) >= 0.5
        ]
    except Exception as exc:
        log.warning(f"Teacher inference failed: {exc}")
        return []


# ── Quick smoke test ─────────────────────────────────────────────────────────
test_entry = (
    "I knew this project was going to fail. I told everyone at the start that the timeline "
    "was unrealistic, and sure enough, here we are behind schedule. It just proves what I've "
    "always believed: management never listens until it's too late. Every bit of news I get "
    "about the delay just confirms what I already knew."
)

student_pred = predict_biases_student(test_entry)
teacher_pred = predict_biases_teacher(test_entry)
print(f"Smoke test entry (confirmation_bias expected):")
print(f"  Student : {student_pred}")
print(f"  Teacher : {teacher_pred}")

In [ ]:
# ── 30-entry human-labeled holdout (2 entries per class × 15 classes) ────────
# These entries are held out from training and used for head-to-head evaluation.
# Each entry has a single ground-truth primary bias (human-curated).

HUMAN_EVAL_SET = [
    # confirmation_bias (2)
    {"text": "I've been reading about why remote work fails and every article I find just confirms what I already suspected — people are less productive at home. I didn't really look into the studies that showed benefits, because honestly they don't match what I see in my own team.", "expected": ["confirmation_bias"]},
    {"text": "The news keeps reporting on crime rates going up and it's exactly what I expected from this neighbourhood. I only save articles that prove my point. Other data just seems cherry-picked to push an agenda.", "expected": ["confirmation_bias"]},
    # attribution_error (2)
    {"text": "My colleague was 20 minutes late to the client call today. Typical — he just doesn't care about his work or other people's time. Meanwhile I was late last week because of a genuine emergency that nobody bothered to consider.", "expected": ["attribution_error"]},
    {"text": "The new intern made three errors in the report. Honestly this generation just doesn't put in the effort. When I mess up it's because the brief was unclear, but these kids just don't try.", "expected": ["attribution_error"]},
    # all_or_nothing (2)
    {"text": "I got one piece of critical feedback in my performance review and it ruined the whole thing for me. There were nine positive comments but if I can't be perfect, what's even the point of trying? Either I'm doing an excellent job or I'm failing — there's no in between.", "expected": ["all_or_nothing"]},
    {"text": "I missed one gym session this week and now my whole fitness routine is ruined. I either do it perfectly or I might as well quit. I can't do anything halfway.", "expected": ["all_or_nothing"]},
    # catastrophizing (2)
    {"text": "My manager asked to speak with me tomorrow and I can't sleep. It must be about the presentation last week. They're probably going to put me on a performance plan, and then I'll lose my job, and I won't be able to pay rent, and I'll have to move back home. My whole career is about to fall apart.", "expected": ["catastrophizing"]},
    {"text": "I felt a sharp pain in my knee during the run. It's probably a serious injury. I'm going to need surgery, never run again, and gain all the weight back. This is the beginning of a long decline.", "expected": ["catastrophizing"]},
    # mind_reading (2)
    {"text": "My friend didn't respond to my text for six hours. She's clearly upset with me about something I said at dinner. I could tell by the way she went quiet that she was judging me. She probably thinks I'm selfish.", "expected": ["mind_reading"]},
    {"text": "The interviewer barely smiled the whole time. He obviously thought I was underqualified. I could see in his eyes he'd already made up his mind before I even finished answering the second question.", "expected": ["mind_reading"]},
    # overgeneralization (2)
    {"text": "My presentation flopped today. I always freeze in front of an audience — it never gets better no matter how much I practice. Every single time I present something important I let everyone down. I'll never be a good communicator.", "expected": ["overgeneralization"]},
    {"text": "I burned dinner again. I always ruin things. Nobody in my family ever compliments my cooking, ever. I'm just not a person who can do domestic things right.", "expected": ["overgeneralization"]},
    # emotional_reasoning (2)
    {"text": "I feel like such an imposter in every meeting. I can't shake the feeling that I don't belong here, which means I probably don't. If I feel this inadequate there must be a real reason for it.", "expected": ["emotional_reasoning"]},
    {"text": "I feel terrified about the surgery even though the surgeon said it's routine. The fear must mean something is really wrong. You don't feel this scared for no reason.", "expected": ["emotional_reasoning"]},
    # should_statements (2)
    {"text": "I should be further along in my career by thirty. I must be more productive every single day. I have to stop wasting time on anything that isn't directly advancing my goals. I should never need to rest.", "expected": ["should_statements"]},
    {"text": "My kids should be grateful for everything I do for them. They must clean their rooms without being asked at their age. A good parent shouldn't have to repeat themselves.", "expected": ["should_statements"]},
    # labeling (2)
    {"text": "I made a mistake on the quarterly numbers and now I can't stop calling myself an idiot in my head. I'm just a screw-up, plain and simple. People like me don't succeed in finance.", "expected": ["labeling"]},
    {"text": "My ex is just a narcissist. Everything she did makes sense once you accept that she is simply a toxic person. There's nothing more to understand about her.", "expected": ["labeling"]},
    # personalization (2)
    {"text": "The team didn't hit the quarterly target and I feel responsible. If I had pushed harder and worked more weekends we would have made it. I should have spotted the market shift earlier. This is on me.", "expected": ["personalization"]},
    {"text": "My parents' divorce happened when I was nine. I've always felt like it was somehow my fault — if I'd been a better kid maybe they would have stayed together and tried harder.", "expected": ["personalization"]},
    # availability_bias (2)
    {"text": "Three people in my building tested positive for a rare condition last month. Now I'm convinced it's really common and I should probably get tested. Every time I hear about it in the news it feels more and more like an epidemic.", "expected": ["availability_bias"]},
    {"text": "A friend of mine got badly burned by a startup investment last year and I can't stop thinking about that when I look at my own portfolio. Tech startups just feel incredibly risky right now even though the data says otherwise.", "expected": ["availability_bias"]},
    # anchoring_bias (2)
    {"text": "The first salary I was quoted for this role was $70k. Even though I've since learned the market rate is $90k, the negotiation keeps pulling me back toward $70k. I feel guilty asking for more than that original number.", "expected": ["anchoring_bias"]},
    {"text": "The house was listed at $850k so when they dropped it to $780k I thought it was a bargain. My partner pointed out that comparable homes sell for $720k, but I keep thinking we're saving $70k off the list price.", "expected": ["anchoring_bias"]},
    # dunning_kruger (2)
    {"text": "I took a weekend Python course and I'm honestly shocked at how quickly I understood everything. I feel like I could build almost any application at this point. The senior engineers at work seem to overcomplicate things that I now see are actually quite simple.", "expected": ["dunning_kruger"]},
    {"text": "I've been investing for eight months and I've already outperformed the market. I think I have a genuine talent for spotting value that most analysts miss. Wall Street professionals are overrated.", "expected": ["dunning_kruger"]},
    # sunk_cost_fallacy (2)
    {"text": "I've been in this relationship for four years and invested so much of myself in making it work. Even though I'm consistently unhappy and we want different things, I can't walk away after putting in all that time and energy.", "expected": ["sunk_cost_fallacy"]},
    {"text": "I'm 60% through a 400-page book I'm not enjoying at all. But I've read so much of it already that I feel I have to finish it. Stopping now would mean all that time was wasted.", "expected": ["sunk_cost_fallacy"]},
    # fundamental_attribution (2)
    {"text": "The customer service rep was rude and unhelpful. She's just one of those people who doesn't care about her job. It never crossed my mind that she might be on hour nine of a ten-hour shift dealing with angry customers all day.", "expected": ["fundamental_attribution"]},
    {"text": "The driver who cut me off was just an aggressive, selfish person. People like that shouldn't be on the road. I didn't consider that they might be rushing to a hospital or hadn't slept.", "expected": ["fundamental_attribution"]},
]

assert len(HUMAN_EVAL_SET) == 30, f"Expected 30 entries, got {len(HUMAN_EVAL_SET)}"
assert all(e["expected"] and all(b in LABEL2ID for b in e["expected"]) for e in HUMAN_EVAL_SET), \
    "Some entries have invalid bias IDs in 'expected'"

# Verify coverage: all 15 classes represented
covered = set(b for e in HUMAN_EVAL_SET for b in e["expected"])
missing = set(BIAS_LABELS) - covered
assert not missing, f"Missing classes in eval set: {missing}"

print(f"Human eval set: {len(HUMAN_EVAL_SET)} entries, {len(covered)}/{N_CLASSES} classes covered")

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def evaluate_model(predictor_fn, eval_set: list[dict], name: str) -> dict:
    """
    Run predictor_fn on all entries and compute per-class precision/recall/F1.
    Returns a dict with per-class and macro metrics.
    """
    print(f"\nEvaluating: {name} ({len(eval_set)} entries)...")

    y_true = np.zeros((len(eval_set), N_CLASSES), dtype=int)
    y_pred = np.zeros((len(eval_set), N_CLASSES), dtype=int)

    for i, entry in enumerate(eval_set):
        # Ground truth
        for b in entry["expected"]:
            if b in LABEL2ID:
                y_true[i, LABEL2ID[b]] = 1
        # Prediction
        try:
            predicted = predictor_fn(entry["text"])
        except Exception as exc:
            log.warning(f"Prediction failed for entry {i}: {exc}")
            predicted = []
        for b in predicted:
            if b in LABEL2ID:
                y_pred[i, LABEL2ID[b]] = 1

        if (i + 1) % 5 == 0:
            print(f"  {i+1}/{len(eval_set)} entries processed")
        if name == "Claude Haiku (teacher)":
            time.sleep(0.5)  # rate limit courtesy pause

    # Per-class metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    # Macro averages
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    # Agreement rate: fraction of entries where pred == truth exactly
    exact_match = int(np.all(y_true == y_pred, axis=1).sum())
    agreement_rate = exact_match / len(eval_set)

    results = {
        "name": name,
        "macro_precision": round(float(macro_p), 3),
        "macro_recall":    round(float(macro_r), 3),
        "macro_f1":        round(float(macro_f1), 3),
        "agreement_rate":  round(agreement_rate, 3),
        "per_class": {
            b: {"precision": round(float(precision[i]), 3),
                "recall":    round(float(recall[i]), 3),
                "f1":        round(float(f1[i]), 3),
                "support":   int(support[i])}
            for i, b in enumerate(BIAS_LABELS)
        }
    }

    print(f"  Macro F1: {macro_f1:.3f} | Agreement: {agreement_rate:.1%}")
    return results


# ── Run evaluations ──────────────────────────────────────────────────────────
student_results = evaluate_model(predict_biases_student, HUMAN_EVAL_SET, "QLoRA student (Qwen2.5-3B)")
teacher_results = evaluate_model(predict_biases_teacher, HUMAN_EVAL_SET, "Claude Haiku (teacher)")

# Save results
all_results = {"student": student_results, "teacher": teacher_results}
with open(RESULTS_PATH, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to {RESULTS_PATH}")

In [ ]:
# ── Head-to-head results table ───────────────────────────────────────────────
print("\n" + "=" * 90)
print(f"HEAD-TO-HEAD EVALUATION: 30-entry human-labeled holdout")
print("=" * 90)
print(f"{'Bias Class':<32s} {'Student F1':>10s} {'Teacher F1':>10s} {'Delta':>8s}")
print("-" * 90)

deltas = []
for bias in BIAS_LABELS:
    s_f1 = student_results["per_class"][bias]["f1"]
    t_f1 = teacher_results["per_class"][bias]["f1"]
    delta = s_f1 - t_f1
    deltas.append(delta)
    indicator = "↑" if delta > 0 else ("↓" if delta < 0 else "=")
    print(f"  {bias:<30s} {s_f1:>10.3f} {t_f1:>10.3f} {delta:>+7.3f} {indicator}")

print("-" * 90)
s_macro = student_results["macro_f1"]
t_macro = teacher_results["macro_f1"]
print(f"  {'MACRO AVERAGE':<30s} {s_macro:>10.3f} {t_macro:>10.3f} {s_macro-t_macro:>+7.3f}")
print(f"  {'Agreement Rate':<30s} {student_results['agreement_rate']:>10.1%} {teacher_results['agreement_rate']:>10.1%}")
print("=" * 90)

avg_delta = sum(deltas) / len(deltas)
classes_won = sum(1 for d in deltas if d > 0)
classes_tied = sum(1 for d in deltas if d == 0)
print(f"\nSummary: Student wins on {classes_won}/{N_CLASSES} classes, ties {classes_tied}")
print(f"Average per-class delta: {avg_delta:+.3f}")
print(f"\nKey takeaway: 3B QLoRA student vs claude-haiku-4-5-20251001 teacher")
print(f"  Student cost/inference: ~$0 (local or HF Spaces)")
print(f"  Teacher cost/inference: ~$0.0002 (Claude Haiku API)")

if use_wandb:
    wandb.log({
        "eval/student_macro_f1": s_macro,
        "eval/teacher_macro_f1": t_macro,
        "eval/student_agreement": student_results["agreement_rate"],
        "eval/teacher_agreement": teacher_results["agreement_rate"],
    })

In [ ]:
# ── Generated test set evaluation (stratified 10% holdout) ──────────────────
# This measures student performance on the same distribution as training data
# (useful for confirming the model learned the format, not for real-world performance)

print("\nEvaluating student on generated test set (same distribution as training)...")

y_true_gen = np.zeros((len(test_recs), N_CLASSES), dtype=int)
y_pred_gen = np.zeros((len(test_recs), N_CLASSES), dtype=int)

for i, rec in enumerate(test_recs):
    # Ground truth from generated labels
    for b in [rec["primary_bias"]] + rec.get("co_occurring", []):
        if b in LABEL2ID:
            y_true_gen[i, LABEL2ID[b]] = 1
    # Student prediction
    try:
        for b in predict_biases_student(rec["text"]):
            if b in LABEL2ID:
                y_pred_gen[i, LABEL2ID[b]] = 1
    except Exception:
        pass

_, _, f1_per_class, _ = precision_recall_fscore_support(y_true_gen, y_pred_gen, average=None, zero_division=0)
macro_f1_gen = float(f1_per_class.mean())

print(f"Generated test set — Student Macro F1: {macro_f1_gen:.3f} (n={len(test_recs)})")
print("Note: human-labeled holdout F1 is the more honest signal for real-world performance.")

if use_wandb:
    wandb.log({"eval/student_macro_f1_generated_test": macro_f1_gen})

In [ ]:
from huggingface_hub import HfApi

# ── Save adapter locally ─────────────────────────────────────────────────────
adapter_path = OUTPUT_DIR / "final-adapter"
trainer.model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

# Save eval results alongside the adapter
import shutil
shutil.copy(RESULTS_PATH, adapter_path / "eval_results.json")

# Save a model card with results
model_card = f"""---
base_model: {BASE_MODEL}
language: en
license: apache-2.0
tags:
  - cognitive-bias
  - mental-health
  - text-classification
  - qlora
  - peft
---

# Sentio Bias Classifier — QLoRA (Qwen2.5-3B-Instruct)

Teacher-student distillation from Claude Haiku for cognitive bias detection in journal entries.

## Task
Multi-label classification across 15 cognitive bias classes.
Input: first-person journal entry. Output: JSON list of bias IDs.

## Training
- Base: `{BASE_MODEL}`
- QLoRA: 4-bit NF4, LoRA rank 16, alpha 32
- Data: {N_PER_BIAS * N_CLASSES} examples generated by Claude Haiku (50 per class × 15 classes)
- Epochs: 4, LR: 2e-4, batch 8 (effective)

## Evaluation (30-entry human-labeled holdout)
| Model | Macro F1 | Agreement Rate |
|:------|:---------|:---------------|
| QLoRA student | {student_results['macro_f1']:.3f} | {student_results['agreement_rate']:.1%} |
| Claude Haiku teacher | {teacher_results['macro_f1']:.3f} | {teacher_results['agreement_rate']:.1%} |

## Usage
```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import json

base = AutoModelForCausalLM.from_pretrained("{BASE_MODEL}")
model = PeftModel.from_pretrained(base, "<your-hf-username>/{HF_REPO_ID}")
tokenizer = AutoTokenizer.from_pretrained("{BASE_MODEL}")

# Inference: see notebooks/sentio_bias_qlora.ipynb for full example
```
"""

with open(adapter_path / "README.md", "w") as f:
    f.write(model_card)

print(f"Adapter saved to: {adapter_path}")
print(f"Files: {list(adapter_path.iterdir())}")

# ── Push to HuggingFace Hub ──────────────────────────────────────────────────
if HF_TOKEN:
    try:
        api = HfApi(token=HF_TOKEN)
        # Get the username from the token
        user_info = api.whoami()
        hf_username = user_info["name"]
        full_repo_id = f"{hf_username}/{HF_REPO_ID}"

        api.create_repo(repo_id=full_repo_id, exist_ok=True, private=False)
        api.upload_folder(
            folder_path=str(adapter_path),
            repo_id=full_repo_id,
            repo_type="model",
            commit_message=f"QLoRA adapter — Macro F1 student={student_results['macro_f1']:.3f} teacher={teacher_results['macro_f1']:.3f}",
        )
        print(f"\nPushed to HuggingFace Hub: https://huggingface.co/{full_repo_id}")

        if use_wandb:
            wandb.log({"hf_repo": full_repo_id})
    except Exception as exc:
        print(f"HF Hub push failed (continuing): {exc}")
else:
    print("HF_TOKEN not set — skipping Hub push. Adapter saved locally.")

if use_wandb:
    wandb.finish()
    print("W&B run finished")

---
## Results Summary

After running this notebook you should see `eval_results.json` in `/kaggle/working/` with:

```json
{
  "student": {"macro_f1": 0.XX, "agreement_rate": 0.XX, ...},
  "teacher": {"macro_f1": 0.XX, "agreement_rate": 0.XX, ...}
}
```

**Paste these numbers into `sentio-repo/RESULTS.md` under the WS-2 section.**

---

### Production integration plan

The fine-tuned adapter is designed to serve via the Sentio HF Space as a cascade:

```
Journal entry
      ↓
QLoRA student  ──── confidence ≥ 0.7 ──→  return student prediction
      │                                    (zero API cost)
      └── confidence < 0.7 ──→  Claude Haiku fallback
                                            (~$0.0002/entry)
```

Expected outcome: student handles ~75–85% of entries, Haiku covers the rest.
Cost reduction: ~80% vs pure Haiku at scale.

---

### Interview talking points

1. **Why QLoRA over full fine-tuning?**  
   3B model × 4-bit quantization fits in 16 GB T4. LoRA rank 16 adds ~1.3% trainable parameters
   (~40M), updates the model in a low-rank subspace, and has been shown to match full fine-tuning
   on classification tasks at a fraction of the compute.

2. **Why teacher-student distillation?**  
   Claude Haiku has zero-shot capability for bias detection (that's what's in production).  
   Distillation transfers that capability into a local model, removing per-call API latency
   and cost. The student trains on both the task structure (via the format) and the labels
   (via the teacher's annotations).

3. **Why Qwen2.5-3B over DistilBERT (the original sentio-ml model)?**  
   The existing `sentio-ml/train_bias_classifier.py` uses DistilBERT for a classification head.
   Qwen2.5-3B is an instruction-tuned generative model — it produces structured JSON with
   confidence scores rather than a fixed-vocab label, making it easier to extend the taxonomy
   and to output explanations ("span" quotes) alongside predictions.

4. **How do you validate the synthetic training data?**  
   (a) Class distribution check — each class has ≥10 examples  
   (b) Word count check — all entries 20+ words (qualitative realism)  
   (c) Co-occurring bias validation — secondary labels must be valid IDs  
   (d) Primary bias enforced — generator output re-labeled at ingest to prevent hallucination  
   (e) Human-labeled 30-entry eval set — separate from generated data, used as real-world signal

5. **What metrics and why?**  
   Macro F1 (each class weighted equally, not by support) because the 15 classes have equal
   real-world importance — we don't want a dominant class to inflate the score.  
   Agreement rate (exact match) measures how often the model gets the complete label set right.

6. **Production cascade design reasoning**  
   Student model runs locally on HF Spaces (free GPU or CPU inference, ~200ms).  
   Confidence threshold 0.7: below this, fall back to Claude Haiku for the hard cases.  
   This keeps cost proportional to model uncertainty — exactly the cases where the student
   is most likely to be wrong are the ones where the API spend is justified.